# Chương 5 — Ứng dụng thực tế: Phân tích giỏ hàng (Market Basket Analysis)

**Tập dữ liệu:** Groceries (`../data/groceries.txt`)  
**Thuật toán:** H-Mine (cài đặt của nhóm)  
**Mục tiêu:**

> Đọc dữ liệu giao dịch siêu thị thực tế

> Dùng H-Mine khai phá frequent itemsets `sup ≥ minsup`

> Sinh luật kết hợp với `conf ≥ minconf`

> Trình bày top-10 luật theo **lift** và phân tích ý nghĩa kinh doanh

## 1. Load các module nội bộ

In [21]:
import Pkg
Pkg.activate("..") # trỏ về thư mục ngoài (root)
Pkg.instantiate()

  Activating project at `c:\Users\ADMIN\Documents\Chuyen Nganh\HK2\Khai thác dữ liệu\Lab2 - DM\Lab2-DataMining-HMine`


In [22]:
using Printf

# Thay vì gọi lắt nhắt mấy file nhỏ, mình gọi thẳng file lớn chứa toàn bộ hàm luôn
include("../src/market_basket_groceries.jl")

using .Structures
using .Rules

println("✓ Đã load xong!")

✓ Đã load xong!


## 2. Tập dữ liệu Groceries

File `data/groceries.txt` chứa lịch sử giao dịch của một siêu thị thực tế.  
**Định dạng:** Mỗi dòng = 1 giao dịch, các sản phẩm cách nhau bằng dấu phẩy.

```
citrus fruit,semi-finished bread,margarine,ready soups
tropical fruit,yogurt,coffee
whole milk
...
```

In [23]:
# Đọc dữ liệu
# Lưu ý: Đường dẫn tương đối từ thư mục notebooks/ là ../data/
GROCERIES_PATH = "../data/groceries/groceries.txt"

# Tái sử dụng hàm read_groceries từ file market_basket_groceries.jl
db_encoded, item_to_id, id_to_item = read_groceries(GROCERIES_PATH)

n_trans = length(db_encoded)
n_items = length(item_to_id)

println("Số giao dịch  : $n_trans")
println("Số items duy nhất : $n_items")

# Thống kê độ dài giao dịch
lengths = [length(t) for t in db_encoded]
println("Độ dài TB/giao dịch : $(round(sum(lengths)/length(lengths), digits=2)) items")
println("Độ dài min/max      : $(minimum(lengths)) / $(maximum(lengths)) items")

Số giao dịch  : 9835
Số items duy nhất : 169
Độ dài TB/giao dịch : 4.41 items
Độ dài min/max      : 1 / 32 items


In [24]:
# Xem thử 5 giao dịch đầu tiên
println("5 giao dịch đầu tiên:")
println("-"^50)
for (i, trans) in enumerate(db_encoded[1:5])
    items = join([id_to_item[it] for it in trans], ", ")
    println("TID $i: $items")
end

5 giao dịch đầu tiên:
--------------------------------------------------
TID 1: citrus fruit, semi-finished bread, margarine, ready soups
TID 2: tropical fruit, yogurt, coffee
TID 3: whole milk
TID 4: pip fruit, yogurt, cream cheese, meat spreads
TID 5: other vegetables, whole milk, condensed milk, long life bakery product


## 3. Khai phá Frequent Itemsets bằng H-Mine

**Tham số:**
- `minsup = 1%` tương đương 99 giao dịch — đủ ý nghĩa thống kê, không bỏ sót sản phẩm ít phổ biến

**Luồng xử lý của `run_hmine_collect`:**
1. Đếm tần suất, lọc F-list theo minsup
2. Xây dựng HeaderTable và filtered database
3. Gọi `mine_h_opt` (H-Mine tối ưu), capture output vào buffer
4. Parse buffer → `Dict{Vector{Int}, Int}` (itemset → count)

In [25]:
# Thiết lập tham số
MINSUP_PERCENT = 1.0
min_sup_value  = max(1, Int(ceil(n_trans * MINSUP_PERCENT / 100.0)))

println("minsup = $(MINSUP_PERCENT)% → $min_sup_value giao dịch")

# Tái sử dụng hàm run_hmine_collect từ file market_basket_groceries.jl
t0 = time()
freq_itemsets = run_hmine_collect(db_encoded, min_sup_value)
elapsed = round(time() - t0, digits=3)

println("Thời gian H-Mine   : $(elapsed) giây")
println("Frequent itemsets  : $(length(freq_itemsets))")

# Ghi file frequent itemsets
output_items_path = "../data/groceries/frequent_itemsets.txt"

print("Đang ghi kết quả ra file... ")
open(output_items_path, "w") do f
    println(f, "=========================================")
    println(f, "           FREQUENT ITEMSETS            ")
    println(f, "=========================================")
    
    # Sắp xếp itemsets theo tần suất giảm dần
    sorted_itemsets = sort(collect(freq_itemsets), by = x -> x[2], rev=true)
    
    for (itemset, count) in sorted_itemsets
        # Ánh xạ từ ID sang tên sản phẩm thực tế
        item_names = [get(id_to_item, i, "item_$i") for i in itemset]
        println(f, "{ " * join(item_names, ", ") * " } #SUP: $count")
    end
end
println("Xong!")
println("✓ Đã lưu file tại: $output_items_path")

minsup = 1.0% → 99 giao dịch
Thời gian H-Mine   : 1.015 giây
Frequent itemsets  : 333
Đang ghi kết quả ra file... Xong!
✓ Đã lưu file tại: ../data/groceries/frequent_itemsets.txt


In [26]:
# Thống kê phân bổ theo kích thước itemset
by_size = Dict{Int,Int}()
for (itemset, _) in freq_itemsets
    k = length(itemset)
    by_size[k] = get(by_size, k, 0) + 1
end

println("Phân bổ frequent itemsets theo kích thước:")
println("-"^35)
for k in sort(collect(keys(by_size)))
    println("  $(k)-itemset : $(by_size[k])")
end
println("-"^35)
println("  Tổng cộng : $(length(freq_itemsets))")

Phân bổ frequent itemsets theo kích thước:
-----------------------------------
  1-itemset : 88
  2-itemset : 213
  3-itemset : 32
-----------------------------------
  Tổng cộng : 333


In [27]:
# Top-10 frequent itemsets phổ biến nhất (theo count)
sorted_freq = sort(collect(freq_itemsets), by = x -> x[2], rev = true)

println("Top-10 frequent itemsets phổ biến nhất:")
println("-"^55)
@printf("%-4s  %-35s  %s\n", "#", "Itemset", "Count")
println("-"^55)
for (i, (itemset, cnt)) in enumerate(sorted_freq[1:10])
    names = join([id_to_item[it] for it in itemset], ", ")
    @printf("%-4d  %-35s  %d\n", i, "{"*names*"}", cnt)
end

Top-10 frequent itemsets phổ biến nhất:
-------------------------------------------------------
#     Itemset                              Count
-------------------------------------------------------
1     {whole milk}                         2513
2     {other vegetables}                   1903
3     {rolls/buns}                         1809
4     {soda}                               1715
5     {yogurt}                             1372
6     {bottled water}                      1087
7     {root vegetables}                    1072
8     {tropical fruit}                     1032
9     {shopping bags}                      969
10    {sausage}                            924


## 4. Sinh Association Rules

Từ tập frequent itemsets, sinh các luật $X \Rightarrow Y$ thỏa:
$$\text{confidence}(X \Rightarrow Y) = \frac{\text{sup}(X \cup Y)}{\text{sup}(X)} \geq \text{minconf}$$

Sau đó tính thêm:
$$\text{lift}(X \Rightarrow Y) = \frac{\text{conf}(X \Rightarrow Y)}{\text{sup}(Y)}$$
$$\text{conviction}(X \Rightarrow Y) = \frac{1 - \text{sup}(Y)}{1 - \text{conf}(X \Rightarrow Y)}$$

**Ý nghĩa các độ đo:**
* **Support**: Tần suất xuất hiện đồng thời của $X$ và $Y$. Lọc ra các luật có ý nghĩa thống kê, tránh kết luận từ quá ít quan sát.
* **Confidence**: Xác suất có điều kiện $P(Y \mid X)$. Đo mức độ tin cậy của luật.
* **Lift**: Tỉ số giữa confidence thực tế và confidence kỳ vọng nếu $X$, $Y$ độc lập. Lift $> 1$ cho thấy $X$ và $Y$ có quan hệ thuận chiều; lift $= 1$ là độc lập; lift $< 1$ là quan hệ nghịch chiều. Lift là độ đo quan trọng nhất để xếp hạng luật vì loại trừ ảnh hưởng của sản phẩm phổ biến.
* **Conviction**: Đo mức độ phụ thuộc của $Y$ vào $X$. Conviction càng cao thì luật càng chắc chắn, ít phụ thuộc vào ngẫu nhiên.

**Tham số:** `minconf = 20%` — cho phép khám phá luật đa dạng, lift sẽ lọc và xếp hạng thêm.  
Dùng `single_consequent = false` để sinh cả luật có vế phải nhiều item (ví dụ: `{butter} => {whole milk, other vegetables}`).

In [28]:
MINCONF = 0.2

# Chỉ định rõ hàm được gọi từ module Rules
rules = Rules.generate_rules(
    freq_itemsets,
    id_to_item,
    n_trans,
    MINCONF;
    single_consequent = false
)

println("minconf = $(Int(MINCONF*100))%")
println("Số luật kết hợp sinh được : $(length(rules))")

# Thống kê
all_lifts = [r.lift for r in rules]
all_confs = [r.confidence for r in rules]

println()
println("Thống kê toàn bộ tập luật:")
println("-"^40)
@printf("  Lift  — trung bình: %.3f  |  cao nhất: %.3f\n",
        sum(all_lifts)/length(all_lifts), maximum(all_lifts))
@printf("  Conf  — trung bình: %.1f%%  |  cao nhất: %.1f%%\n",
        sum(all_confs)/length(all_confs)*100, maximum(all_confs)*100)

# Ghi file luật kết hợp
output_rules_path = "../data/groceries/association_rules.txt"

print("\nĐang ghi kết quả luật kết hợp ra file... ")
open(output_rules_path, "w") do f
    # Ghi Thống kê tổng quát
    println(f, "=========================================")
    println(f, "           THỐNG KÊ TỔNG QUÁT           ")
    println(f, "=========================================")
    println(f, "Tổng số luật         : $(length(rules))")
    println(f, "Lift trung bình      : $(round(sum(all_lifts)/length(all_lifts), digits=3))")
    println(f, "Lift cao nhất        : $(round(maximum(all_lifts), digits=3))")
    println(f, "Confidence trung bình: $(round(sum(all_confs)/length(all_confs)*100, digits=1))%")
    println(f, "Confidence cao nhất  : $(round(maximum(all_confs)*100, digits=1))%\n")

    # Ghi toàn bộ luật kết hợp
    println(f, "=========================================")
    println(f, "           ASSOCIATION RULES            ")
    println(f, "=========================================")
    
    # Sắp xếp luật theo Lift giảm dần
    sorted_rules = sort(rules, by = r -> r.lift, rev=true)
    
    for r in sorted_rules
        ant_str = "{" * join(r.antecedent, ", ") * "}"
        con_str = "{" * join(r.consequent, ", ") * "}"
        
        support_val = round(r.support, digits=4)
        conf_val    = round(r.confidence, digits=4)
        lift_val    = round(r.lift, digits=4)
        conv_str    = isinf(r.conviction) ? "inf" : string(round(r.conviction, digits=4))
        
        println(f, "$ant_str => $con_str")
        println(f, "   [ support: $support_val | confidence: $conf_val | lift: $lift_val | conviction: $conv_str ]")
    end
end
println("Xong!")
println("✓ Đã lưu file tại: $output_rules_path")

minconf = 20%
Số luật kết hợp sinh được : 234

Thống kê toàn bộ tập luật:
----------------------------------------
  Lift  — trung bình: 1.805  |  cao nhất: 3.295
  Conf  — trung bình: 33.1%  |  cao nhất: 58.6%

Đang ghi kết quả luật kết hợp ra file... Xong!
✓ Đã lưu file tại: ../data/groceries/association_rules.txt


## 5. Top-10 luật theo Lift

**Lift** là độ đo xếp hạng tốt nhất vì nó loại trừ ảnh hưởng của sản phẩm phổ biến:
- Lift > 1: X và Y có xu hướng mua cùng nhau (quan hệ thuận chiều)
- Lift = 1: X và Y độc lập
- Lift < 1: X và Y ít đi cùng nhau

In [29]:
TOP_K = 10

# Thêm Rules. vào trước hàm top_rules_by_lift
top_rules = Rules.top_rules_by_lift(rules, TOP_K)

Rules.print_rules(
    top_rules;
    title = "TOP-$TOP_K ASSOCIATION RULES THEO LIFT " *
            "(minsup=$(MINSUP_PERCENT)%, minconf=$(Int(MINCONF*100))%)"
)


  TOP-10 ASSOCIATION RULES THEO LIFT (minsup=1.0%, minconf=20%)
#   Antecedent (X)                      => Consequent (Y)                   Support   Confidence  Lift    Conviction
---------------------------------------------------------------------------------------------------------
1   {citrus fruit, other vegetables}    => {root vegetables}                0.0104    0.3592      3.2950    1.3904
2   {yogurt, other vegetables}          => {whipped/sour cream}             0.0102    0.2342      3.2671    1.2122
3   {tropical fruit, other vegetables}  => {root vegetables}                0.0123    0.3428      3.1448    1.3557
4   {beef}                              => {root vegetables}                0.0174    0.3314      3.0404    1.3326
5   {citrus fruit, root vegetables}     => {other vegetables}               0.0104    0.5862      3.0296    1.9491
6   {tropical fruit, root vegetables}   => {other vegetables}               0.0123    0.5845      3.0210    1.9412
7   {whole milk, other

## 6. Phân tích ý nghĩa kinh doanh

### 6.1 Nhóm luật về rau củ (Luật 1, 3, 4, 5, 6, 7, 8)

Bảy trong mười luật đầu xoay quanh ba nhóm sản phẩm:
**root vegetables** (cà rốt, khoai tây, củ cải), **other vegetables** (rau xanh tổng hợp), và **citrus/tropical fruit** (trái cây có múi/nhiệt đới).

| # | Luật | Lift | Nhận xét |
|---|------|------|-----------|
| 1 | {citrus fruit, other vegetables} ⟹ {root vegetables} | 3.295 | Lift cao nhất — nhóm khách ăn uống lành mạnh |
| 4 | {beef} ⟹ {root vegetables} | 3.040 | 1/3 khách mua thịt bò đều mua thêm rau củ — nguyên liệu nấu hầm/súp |
| 5 | {citrus fruit, root vegetables} ⟹ {other vegetables} | 3.030 | Conf = 58.6% — cao nhất trong toàn bộ tập luật |

### 6.2 Nhóm luật về sản phẩm từ sữa (Luật 2, 9, 10)

| # | Luật | Lift | Nhận xét |
|---|------|------|-----------|
| 2 | {yogurt, other vegetables} ⟹ {whipped/sour cream} | 3.267 | Thói quen ẩm thực Tây Âu — kem chua dùng làm sốt |
| 9 | {butter} ⟹ {whole milk, other vegetables} | 2.771 | Bộ nguyên liệu nấu ăn quen thuộc |
| 10 | {whole milk, curd} ⟹ {yogurt} | 2.761 | Cluster sản phẩm từ sữa — khách hàng trung thành |

## 7. Nhận xét tổng hợp và đề xuất ứng dụng

### Hai nhóm khách hàng nhận diện được

**Nhóm rau củ – thịt** (Luật 1, 3, 4, 5, 6, 7): Mua rau củ, trái cây tươi và thịt theo mẫu nguyên liệu bữa tối. Chiếm đa số trong dữ liệu.

**Nhóm sản phẩm từ sữa** (Luật 2, 9, 10): Mua các sản phẩm từ sữa kết hợp với nhau. Khách hàng trung thành, mua định kỳ.

### Đề xuất ứng dụng

1. **Bố trí kệ hàng**: Đặt *root vegetables*, *other vegetables* và *citrus/tropical fruit* trong cùng khu vực (Luật 1, 3, 5, 6). Nhóm sản phẩm từ sữa trên cùng dãy tủ lạnh (Luật 10).

2. **Combo khuyến mãi**: Gói thịt bò kèm rau củ (Luật 4), gói sữa tươi kèm sữa chua và phô mai tươi (Luật 10).

3. **Gợi ý sản phẩm**: Khi khách thêm *beef* vào giỏ → gợi ý *root vegetables* (Luật 4, conf = 33.1%).

4. **Quản lý tồn kho**: Lập lịch nhập kho đồng bộ cho các sản phẩm trong cùng luật.

5. **Phân khúc khách hàng**: Xây dựng chương trình tích điểm riêng cho nhóm rau củ–thịt và nhóm sản phẩm từ sữa.

In [30]:
# Tóm tắt toàn bộ kết quả chương 5
println("=" ^ 55)
println("  TÓM TẮT KẾT QUẢ — CHƯƠNG 5")
println("=" ^ 55)
@printf("  Tập dữ liệu       : Groceries\n")
@printf("  Giao dịch         : %d\n", n_trans)
@printf("  Items duy nhất    : %d\n", n_items)
@printf("  minsup            : %.1f%% (%d giao dịch)\n", MINSUP_PERCENT, min_sup_value)
@printf("  minconf           : %.0f%%\n", MINCONF * 100)
println("-" ^ 55)
@printf("  Frequent itemsets : %d\n", length(freq_itemsets))
@printf("  Association rules : %d\n", length(rules))
println("-" ^ 55)
@printf("  Lift cao nhất     : %.4f\n", maximum(all_lifts))
@printf("  Lift trung bình   : %.3f\n", sum(all_lifts)/length(all_lifts))
@printf("  Conf cao nhất     : %.1f%%\n", maximum(all_confs)*100)
@printf("  Conf trung bình   : %.1f%%\n", sum(all_confs)/length(all_confs)*100)
println("=" ^ 55)

  TÓM TẮT KẾT QUẢ — CHƯƠNG 5
  Tập dữ liệu       : Groceries
  Giao dịch         : 9835
  Items duy nhất    : 169
  minsup            : 1.0% (99 giao dịch)
  minconf           : 20%
-------------------------------------------------------
  Frequent itemsets : 333
  Association rules : 234
-------------------------------------------------------
  Lift cao nhất     : 3.2950
  Lift trung bình   : 1.805
  Conf cao nhất     : 58.6%
  Conf trung bình   : 33.1%
